# 02 道路小时交通量预测

监测小组需要预测未来1、3、6小时的断面交通量。你要比较简单基线与岭回归，提交模型选择备忘录。目标是建立可靠评估，不要求复杂模型一定胜出。

建议工作量：10—14小时。本工作本是项目起点，默认程序的输出不是完整作业答案。

## 最终成果
- 预测时点与可用特征清单
- 基线和模型的测试误差表
- 高峰与非高峰误差分析
- 模型选择备忘录及复现配置


## 数据与范围

[UCI Metro Interstate Traffic Volume](https://archive.ics.uci.edu/dataset/492/metro+interstate+traffic+volume)

CC BY 4.0，John Hogue (2019)

2017年训练，2018年1—6月验证，7—9月测试。工作台重算预先生成的预测结果的子组误差，不在浏览器重新训练岭回归。

- 预测的是小时交通量，不是车速或拥堵等级。
- 按时间切分数据，测试集不能用于选参数。
- 未来天气实测值不能当作预测时点已知信息。
- 不同模型在同一组有效样本上比较。


In [ ]:
from pathlib import Path
import sys, json
candidates = [Path.cwd(), Path.cwd().parent]
ROOT = next((p for p in candidates if (p / "python" / "analyze.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the project-kit folder or its notebooks folder")
sys.path.insert(0, str(ROOT / "python"))
from analyze import run
OUTPUT = ROOT / "outputs" / "student"
OUTPUT.mkdir(parents=True, exist_ok=True)


## 参考分析与口径检查

先运行一次，解释每个指标的分母和单位。打开源码确认数据筛选规则。预测项目这里读取冻结模型的结果，不重新训练。


In [ ]:
project = "forecast"
config = {
  "horizon": "1",
  "model": "ridge",
  "period": "all",
  "month": "all"
}
result = run(project, config, ROOT / "data")
print(json.dumps(result["metrics"], ensure_ascii=False, indent=2))
print("Source SHA-256:", result["sourceSha"])


## 对照实验

以下配置提供一个可运行起点。说明每次只改变了什么，以及还存在哪些混杂条件。增加你自己的对照，不只重复默认结果。


In [ ]:
comparisons = [
  {
    "horizon": "1",
    "model": "calendar"
  },
  {
    "horizon": "1",
    "model": "ridge"
  },
  {
    "horizon": "6",
    "model": "ridge",
    "period": "peak"
  }
]
experiments = []
for change in comparisons:
    trial = run(project, {**config, **change}, ROOT / "data")
    experiments.append({"project": project, "config": trial["config"], "metrics": trial["metrics"], "sourceSha": trial["sourceSha"]})
    print(json.dumps(experiments[-1], ensure_ascii=False))
(OUTPUT / (project + "-comparison.json")).write_text(json.dumps(experiments, ensure_ascii=False, indent=2), encoding="utf-8")


## 01 评估协议

**先确定预测时点**

解释预测起点与目标时刻。列出1、3、6小时任务可使用的字段，指出禁止使用的信息。

阶段成果：时间切分和特征可获得性说明。

### 我的证据与解释

在这里填写自己的分析，引用结果行、实验参数或图表。


## 02 基线比较

**建立公平比较**

比较上一已知值、前日同期、上周同期与训练期星期小时均值。哪个基线在哪些条件下更有价值？

阶段成果：至少对照2种基线，注明共同有效样本数。

### 我的证据与解释

在这里填写自己的分析，引用结果行、实验参数或图表。


## 03 建模与误差

**检查模型的实际收益**

比较岭回归与最佳基线，切换预测跨度和高峰筛选。保存至少3次实验，检查平均值是否掩盖子组误差。

阶段成果：包含MAE、RMSE、平均有符号误差及高峰子组。

### 我的证据与解释

在这里填写自己的分析，引用结果行、实验参数或图表。


## 04 模型审查

**形成审慎的选择**

回到Python查看特征与验证期选参。模型在哪些时段失效？部署前还需哪些信息和检验？

阶段成果：可复现配置、失败案例和模型选择备忘录。

### 我的证据与解释

在这里填写自己的分析，引用结果行、实验参数或图表。


## 深入分析

- 在保持时间协议不变的条件下增加泊松回归或非线性模型。
- 用更早月份做多折滚动验证，但不要用最终测试月反复调参。

修改 python/analyze.py 或 python/prepare_data.py 前，先复制为自己的版本并保留数据与参数来源。


## 提交前自查

- [ ] 训练、验证、测试没有交叉。
- [ ] 比较模型使用相同样本和单位。
- [ ] 结论包含高峰误差与失败条件，不仅报告最小MAE。

报告应包含研究问题、方法对照、发现、局限、源数据哈希和复现命令。请附代码、配置、结果CSV。阶段文字与实验次数不自动换算成绩。
